In [ ]:
# Open-source sanitization: cached outputs cleared to remove credential and session traces.
import wandb
import pandas as pd
import os
# Open-source sanitization: hardcoded WANDB_API_KEY removed.

api = wandb.Api()
# run_ids = ['kogor9ti', '9olwcijd', 'b82gvcxo', 'in5yd3pw'] # coder-q3_4b-32k-bs32-mbs32-n8-no_rej
# run_ids = ['0cjrnk6m', 'q6cb0ufy', 'ra049ptj'] # coder-q3_4b_b_acecode1500-32k-bs16-mbs16-n16-no_rej-t0_85
# run_ids = ['fz2v75zs'] # coder-q3_4b_b_acecode1500-32k-bs16-mbs16-n16-no_rej-t0_7
# run_ids = ['dapkxt8w'] # coder-q3_4b_b_acecode1500-32k-bs16-mbs16-n16-no_rej-t0_6
# run_ids = ['5i7nnpsu'] # klear
run_ids = ['kxdvytbk', 'wx3cfyyf'] # coder-r1d_7b-bs32-mbs32-n8-no_rej-8gpu
key_name = 'critic/score/mean'

run = api.run("wangshouli0801-Harbin Institute of Technology/reward_hacking_mre/" + run_ids[0])
history = run.history(keys=[key_name])
history.set_index("_step", inplace=True)

for run_id in run_ids[1:]:
    run = api.run("wangshouli0801-Harbin Institute of Technology/reward_hacking_mre/" + run_id)
    history_ = run.history(keys=[key_name])
    history_.set_index("_step", inplace=True)
    history = history.combine_first(history_)

history.reset_index(inplace=True)

In [ ]:
import numpy as np

# 用一条直线拟合 history[key_name]，离直线越远的点权重越小（稳健拟合）
x = history["_step"].values.astype(float)
y = history[key_name].values.astype(float)
mask = ~np.isnan(y)
x, y = x[mask], y[mask]

def weighted_polyfit_1d(x, y, w):
    """加权一元线性拟合: min sum(w * (y - (a*x+b))^2)"""
    X = np.column_stack([x, np.ones_like(x)])
    s = np.sqrt(w)
    Xw = s[:, None] * X
    yw = s * y
    coef, _, _, _ = np.linalg.lstsq(Xw, yw, rcond=None)
    return coef[0], coef[1]

# 迭代重加权：先普通拟合，再按残差赋权（残差大的权小），再拟合
slope, intercept = np.polyfit(x, y, 1)
for _ in range(10):  # 迭代几次即可稳定
    resid = np.abs(y - (slope * x + intercept))
    scale = np.median(resid) + 1e-8  # 用中位数做尺度，避免被极端值拉垮
    w = 1 / (1 + (resid / scale) ** 2)  # 离得越远权重越小
    slope, intercept = weighted_polyfit_1d(x, y, w)

print(f"拟合直线（离直线远的点权小）: y = {slope:.6e} * x + {intercept:.6f}")
print(f"斜率 slope = {slope:.6e}  (每 step 平均变化量)")

# 画散点 + 拟合直线
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.scatter(x, y, s=8, alpha=0.6, label=key_name)
x_line = np.array([x.min(), x.max()])
plt.plot(x_line, slope * x_line + intercept, "r-", lw=2, label=f"slope={slope:.4e})")
plt.xlabel("_step")
plt.ylabel(key_name)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 滑动窗口：窗口大小 window，步长 stride
window = 10   # 每个窗口包含的 step 数
stride = 10   # 窗口每次移动的 step 数（stride < window 时重叠）

chunk_means = []
max_step = history["_step"].max()
i = 0
while True:
    start = 1 + i * stride
    end = start + window
    if start > max_step:
        break
    print(start, end=' ')
    mask = (history["_step"] >= start) & (history["_step"] < end)
    if mask.any():
        # print(history.loc[mask, key_name])
        # print("#" * 50)
        chunk_means.append(history.loc[mask, key_name].mean())
    # if end > max_step:
    #     break
    i += 1

print(f"共 {len(chunk_means)} 个窗口，window={window}, stride={stride}")
print("各窗口平均值:", chunk_means)

In [ ]:
import matplotlib.pyplot as plt

# 横轴：组号（或可用每组对应的 step 区间中点）
x = range(len(chunk_means))
plt.figure(figsize=(10, 5))
plt.plot(x, chunk_means, "o-", linewidth=2, markersize=8)
plt.xlabel(f"window index (window={window}, stride={stride})")
plt.ylabel("mean of critic/score/mean")
plt.title("mean of critic/score/mean by group")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()